# Stage 1: Non-Instruction Fine-Tuning for AEO/GEO (Unsloth)

This notebook adapts a base model to AEO/GEO domain language before instruction tuning.

Pipeline:
1. Read raw domain text from PDF paragraph-by-paragraph.
2. Clean and chunk text.
3. Evaluate base model on 10 domain prompts (before training).
4. Train with QLoRA on raw text.
5. Save adapter/model.
6. Test post-training behavior on the same prompts.

In [7]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
import os
os.chdir('/content/drive/MyDrive/sft')



# Get the current working directory
print(os.getcwd())

/content/drive/MyDrive/sft


In [10]:
!pip install -r requirements.txt

In [11]:
import json
import os
import random
import re
from dataclasses import dataclass
from pathlib import Path
from typing import List

import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from pypdf import PdfReader

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = ROOT / "data"
REPORTS_DIR = ROOT / "reports"
MODELS_DIR = ROOT / "models"

PDF_PATH = DATA_DIR / "aeo_geo_fine_tuning_corpus.pdf"
BASE_MODEL_ID = "unsloth/Llama-3.2-1B-Instruct-bnb-4bit"
FALLBACK_MODEL_ID = "meta-llama/Llama-3.2-1B-Instruct"
STAGE1_OUT = MODELS_DIR / "stage1_non_instruction_adapter"

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

@dataclass
class TrainConfig:
    max_seq_length: int = 1024
    per_device_train_batch_size: int = 2
    gradient_accumulation_steps: int = 4
    learning_rate: float = 2e-4
    num_train_epochs: int = 1
    warmup_steps: int = 20
    logging_steps: int = 10
    save_steps: int = 100

cfg = TrainConfig()
print(cfg)
print(f"ROOT={ROOT}")
print(f"PDF exists={PDF_PATH.exists()} -> {PDF_PATH}")

TrainConfig(max_seq_length=1024, per_device_train_batch_size=2, gradient_accumulation_steps=4, learning_rate=0.0002, num_train_epochs=1, warmup_steps=20, logging_steps=10, save_steps=100)
ROOT=/content/drive/MyDrive/sft
PDF exists=True -> /content/drive/MyDrive/sft/data/aeo_geo_fine_tuning_corpus.pdf


In [12]:
def extract_pdf_paragraphs(pdf_path: Path) -> List[str]:
    if not pdf_path.exists():
        raise FileNotFoundError(f"PDF not found: {pdf_path}")

    reader = PdfReader(str(pdf_path))
    paragraphs: List[str] = []

    for page_no, page in enumerate(reader.pages, start=1):
        raw = page.extract_text() or ""
        raw = raw.replace("\u00a0", " ")
        blocks = re.split(r"\n\s*\n|\n", raw)

        for block in blocks:
            text = re.sub(r"\s+", " ", block).strip()
            # Remove pure page number lines or tiny numeric noise.
            if re.fullmatch(r"(?:page\s*)?\d+", text.lower()):
                continue
            if len(text) < 30:
                continue
            paragraphs.append(text)

    return paragraphs


def clean_paragraph(text: str) -> str:
    text = re.sub(r"\s+", " ", text).strip()
    text = re.sub(r"^\d+\s+", "", text)
    text = re.sub(r"\s+\d+$", "", text)
    return text


def chunk_text(text: str, chunk_words: int = 140, overlap_words: int = 25) -> List[str]:
    words = text.split()
    if len(words) <= chunk_words:
        return [text]

    chunks = []
    step = max(1, chunk_words - overlap_words)
    for i in range(0, len(words), step):
        chunk = words[i : i + chunk_words]
        if len(chunk) < 40:
            continue
        chunks.append(" ".join(chunk))
    return chunks

paragraphs = extract_pdf_paragraphs(PDF_PATH)
cleaned = [clean_paragraph(p) for p in paragraphs]
cleaned = [p for p in cleaned if len(p) >= 50]

raw_chunks: List[str] = []
for p in cleaned:
    raw_chunks.extend(chunk_text(p, chunk_words=140, overlap_words=25))

train_records = [{"text": t} for t in raw_chunks]
train_ds = Dataset.from_list(train_records)

print(f"Paragraphs: {len(paragraphs)}")
print(f"Cleaned paragraphs: {len(cleaned)}")
print(f"Training chunks: {len(train_ds)}")
print(train_ds[0]["text"][:400])

Paragraphs: 344
Cleaned paragraphs: 312
Training chunks: 312
STRUCTURED UNSUPERVISED FINE-TUNING CORPUS (NON-INSTRUCTIONAL)


In [13]:
questions = [
    "We are launching a new enterprise SaaS tool. How should we structure landing page text and data tables to maximize citation chances in Gemini and ChatGPT Search?",
    "Review an FAQ page with Speakable and Dataset schema. Which microdata fields are commonly missed for AI Overviews and voice assistants?",
    "Perplexity mentions dropped 15 percent this quarter. What framework helps diagnose crawler blocks vs stale stats vs competitor GEO improvements?",
    "Rewrite a blog paragraph using GEO principles with authoritative citations and high information density.",
    "How do I design answer blocks for passage-level retrieval in answer engines?",
    "What authority signals make a brand more citeable in generative search outputs?",
    "How should we measure Share of Model for our brand across major answer engines?",
    "How should we structure comparison tables so LLM retrievers can extract key attributes correctly?",
    "How can we improve entity salience for product pages in AEO and GEO?",
    "How do we balance concise direct answers with deeper context for complex B2B questions?",
]

prompt_template = "You are an AEO and GEO assistant. Answer clearly and practically.\\nQuestion: {q}\\nAnswer:"

BASELINE_MAX_NEW_TOKENS = 180

In [15]:
use_unsloth = True

try:
    from unsloth import FastLanguageModel
except Exception:
    use_unsloth = False

if not use_unsloth:
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from peft import LoraConfig, get_peft_model


def load_model_and_tokenizer(model_id: str):
    if use_unsloth:
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name=model_id,
            max_seq_length=cfg.max_seq_length,
            dtype=None,
            load_in_4bit=True,
        )
        model = FastLanguageModel.get_peft_model(
            model,
            r=16,
            lora_alpha=16,
            lora_dropout=0.0,
            target_modules=[
                "q_proj", "k_proj", "v_proj", "o_proj",
                "gate_proj", "up_proj", "down_proj"
            ],
            use_rslora=False,
        )
        return model, tokenizer

    tokenizer = AutoTokenizer.from_pretrained(model_id)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        device_map="auto",
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    )
    peft_cfg = LoraConfig(
        r=16,
        lora_alpha=16,
        lora_dropout=0.0,
        task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    )
    model = get_peft_model(model, peft_cfg)
    return model, tokenizer


model_id_to_load = BASE_MODEL_ID
model = None
tokenizer = None

try:
    model, tokenizer = load_model_and_tokenizer(model_id_to_load)
except Exception as e:
    print(f"Primary model load failed: {e}")
    print(f"Falling back to {FALLBACK_MODEL_ID}")
    model, tokenizer = load_model_and_tokenizer(FALLBACK_MODEL_ID)

print(f"Loaded model with unsloth={use_unsloth}")


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


🦥 Unsloth Zoo will now patch everything to make training faster!


Unable to import `torchao` Tensor objects. This may affect loading checkpoints serialized with `torchao`
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


==((====))==  Unsloth 2025.11.1: Fast Llama patching. Transformers: 4.57.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.03G [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

Unsloth 2025.11.1 patched 16 layers with 16 QKV layers, 16 O layers and 16 MLP layers.


Loaded model with unsloth=True


In [16]:
def generate_answers(current_model, current_tokenizer, qs: List[str], max_new_tokens: int = 180) -> List[str]:
    answers = []
    current_model.eval()

    for q in qs:
        prompt = prompt_template.format(q=q)
        inputs = current_tokenizer(prompt, return_tensors="pt")
        inputs = {k: v.to(current_model.device) for k, v in inputs.items()}

        with torch.no_grad():
            out = current_model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=0.7,
                top_p=0.9,
                eos_token_id=current_tokenizer.eos_token_id,
                pad_token_id=current_tokenizer.pad_token_id,
            )

        text = current_tokenizer.decode(out[0], skip_special_tokens=True)
        if "Answer:" in text:
            text = text.split("Answer:", 1)[-1].strip()
        answers.append(text)

    return answers


def detect_problem(answer: str) -> str:
    lower = answer.lower()
    if len(answer.split()) < 25:
        return "Too generic and brief; lacks domain-specific depth."
    if not any(k in lower for k in ["schema", "citation", "entity", "retrieval", "table", "evidence", "brand"]):
        return "Insufficient AEO/GEO terminology and weak practical specificity."
    if "it depends" in lower and len(answer.split()) < 70:
        return "Hedged response without actionable framework or steps."
    return "Partially useful but can improve on clarity, structure, and concrete tactics."

baseline_answers = generate_answers(model, tokenizer, questions, max_new_tokens=BASELINE_MAX_NEW_TOKENS)

baseline_df = pd.DataFrame({
    "Question": questions,
    "Base Model Answer": baseline_answers,
    "Problem": [detect_problem(a) for a in baseline_answers],
})

baseline_df.head(10)

,Question,Base Model Answer,Problem
0,We are launching a new enterprise SaaS tool. H...,A - Briefly introduce the new SaaS tool and it...,Insufficient AEO/GEO terminology and weak prac...
1,Review an FAQ page with Speakable and Dataset ...,\nHere are the common microdata fields missed ...,Insufficient AEO/GEO terminology and weak prac...
2,Perplexity mentions dropped 15 percent this qu...,\\n\We use the following 3 frameworks to help ...,"Partially useful but can improve on clarity, s..."
3,Rewrite a blog paragraph using GEO principles ...,Here is a rewritten blog paragraph using GEO p...,"Partially useful but can improve on clarity, s..."
4,How do I design answer blocks for passage-leve...,"AEO and GEO assistants, you can design answer ...","Partially useful but can improve on clarity, s..."
5,What authority signals make a brand more citea...,\nA) \text{ Authority Signals} that include th...,"Partially useful but can improve on clarity, s..."
6,How should we measure Share of Model for our b...,\n\nShare of model across different answer eng...,"Partially useful but can improve on clarity, s..."
7,How should we structure comparison tables so L...,\n\n\begin{tabular}{|p{0.0pt}|p{0.0pt}|p{0.0pt...,"Partially useful but can improve on clarity, s..."
8,How can we improve entity salience for product...,\n\nThe goal is to make the product page more ...,"Partially useful but can improve on clarity, s..."
9,How do we balance concise direct answers with ...,\\nHere are some practical strategies for bala...,Insufficient AEO/GEO terminology and weak prac...


In [17]:
report_path = REPORTS_DIR / "base_model_evaluation.md"

with report_path.open("w", encoding="utf-8") as f:
    f.write("# Base Model Evaluation (Before Fine-Tuning)\n\n")
    f.write("Model target: Llama-3.2-1B (Unsloth 4-bit variant when available).\n\n")
    f.write("| Question | Base Model Answer | Problem |\n")
    f.write("|---|---|---|\n")
    for _, row in baseline_df.iterrows():
        q = str(row["Question"]).replace("|", "\\|").replace("\n", " ")
        a = str(row["Base Model Answer"]).replace("|", "\\|").replace("\n", " ")
        p = str(row["Problem"]).replace("|", "\\|").replace("\n", " ")
        f.write(f"| {q} | {a} | {p} |\n")

print(f"Wrote baseline report to {report_path}")

Wrote baseline report to /content/drive/MyDrive/sft/reports/base_model_evaluation.md


In [18]:
from transformers import TrainingArguments
from trl import SFTTrainer

text_field = "text"

training_args = TrainingArguments(
    output_dir=str(MODELS_DIR / "stage1_runs"),
    per_device_train_batch_size=cfg.per_device_train_batch_size,
    gradient_accumulation_steps=cfg.gradient_accumulation_steps,
    warmup_steps=cfg.warmup_steps,
    num_train_epochs=cfg.num_train_epochs,
    learning_rate=cfg.learning_rate,
    logging_steps=cfg.logging_steps,
    save_steps=cfg.save_steps,
    fp16=not torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False,
    bf16=torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False,
    optim="adamw_8bit",
    lr_scheduler_type="cosine",
    weight_decay=0.01,
    seed=SEED,
    report_to="none",
)

if use_unsloth:
    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=train_ds,
        dataset_text_field=text_field,
        max_seq_length=cfg.max_seq_length,
        packing=True,
        args=training_args,
    )
else:
    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=train_ds,
        dataset_text_field=text_field,
        max_seq_length=cfg.max_seq_length,
        packing=True,
        args=training_args,
    )

trainer.train()

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/312 [00:00<?, ? examples/s]

The model is already on multiple devices. Skipping the move to device specified in `args`.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 312 | Num Epochs = 1 | Total steps = 39
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 11,272,192 of 1,247,086,592 (0.90% trained)


Step,Training Loss
10,5.213600
20,4.861800
30,4.453300


TrainOutput(global_step=39, training_loss=4.710200480925731, metrics={'train_runtime': 53.1927, 'train_samples_per_second': 5.865, 'train_steps_per_second': 0.733, 'total_flos': 34931099099136.0, 'train_loss': 4.710200480925731, 'epoch': 1.0})

In [19]:
STAGE1_OUT.mkdir(parents=True, exist_ok=True)

if hasattr(model, "save_pretrained"):
    model.save_pretrained(str(STAGE1_OUT))
if hasattr(tokenizer, "save_pretrained"):
    tokenizer.save_pretrained(str(STAGE1_OUT))

print(f"Saved Stage 1 adapter/model to {STAGE1_OUT}")

post_answers = generate_answers(model, tokenizer, questions, max_new_tokens=200)
compare_df = pd.DataFrame({
    "Question": questions,
    "Before": baseline_answers,
    "After Stage1": post_answers,
})
compare_df.head(10)

Saved Stage 1 adapter/model to /content/drive/MyDrive/sft/models/stage1_non_instruction_adapter


,Question,Before,After Stage1
0,We are launching a new enterprise SaaS tool. H...,A - Briefly introduce the new SaaS tool and it...,This involves optimizing for the model's searc...
1,Review an FAQ page with Speakable and Dataset ...,\nHere are the common microdata fields missed ...,\nAnswering this question requires a deep unde...
2,Perplexity mentions dropped 15 percent this qu...,\\n\We use the following 3 frameworks to help ...,AEOs (Automated Entity Optimization) and GEO (...
3,Rewrite a blog paragraph using GEO principles ...,Here is a rewritten blog paragraph using GEO p...,Here is a rewritten example of a blog paragrap...
4,How do I design answer blocks for passage-leve...,"AEO and GEO assistants, you can design answer ...",The most effective AEO and GEO strategies invo...
5,What authority signals make a brand more citea...,\nA) \text{ Authority Signals} that include th...,AEOs and GEOs explicitly define the semantic s...
6,How should we measure Share of Model for our b...,\n\nShare of model across different answer eng...,"\nAEOs and GEOs must prioritize a broad, keywo..."
7,How should we structure comparison tables so L...,\n\n\begin{tabular}{|p{0.0pt}|p{0.0pt}|p{0.0pt...,AEOs and GEOs should be structured with a clea...
8,How can we improve entity salience for product...,\n\nThe goal is to make the product page more ...,AEO and GEO engines are designed to generate c...
9,How do we balance concise direct answers with ...,\\nHere are some practical strategies for bala...,"\nAEOs and GEOs must leverage a dense, high-ve..."
